In [1]:
# activity data
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient

In [2]:
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

In [3]:
sleep_df1 = gen_date_col(data_dict['Sleep'],tcol='timestamp')
sleep_df1

# The sleep bouts in the given 22:00 to 06:00 period (22:00-24:00 in the previous day, and 0:00-6:00 in the same day)

In [4]:
from datetime import time, timedelta
sleep_df1 = sleep_df1[
    (df['timestamp'].dt.time < time(6, 0)) | 
    (df['timestamp'].dt.time >= time(22, 0))
]

sleep_df1

In [5]:
sleep_id_array = sleep_df1.patient_id.unique()
sleep_id_array

In [6]:
from datetime import time, timedelta
# Define function for custom sleep_date
def get_sleep_date(ts):
    t = ts.time()
    if time(0, 0) <= t < time(22, 0):
        return ts.date()
    else:
        return (ts + timedelta(days=1)).date()

# Apply the function
sleep_df1['sleep_date'] = sleep_df1['timestamp'].apply(get_sleep_date)
sleep_df1

In [7]:
third_order = [
    (['LIGHT'], ['AWAKE']),
    (['DEEP'], ['AWAKE', 'LIGHT']),
    (['REM'], ['AWAKE', 'LIGHT', 'DEEP'])
]

second_order = [
    (['LIGHT', 'DEEP'], ['AWAKE', 'LIGHT']),
    (['LIGHT', 'REM'], ['AWAKE', 'LIGHT', 'DEEP']),
    (['deep', 'REM'], ['AWAKE', 'LIGHT', 'DEEP'])
]

first_order = [
    (['LIGHT', 'DEEP', 'REM'], ['AWAKE', 'LIGHT', 'DEEP'])
]

# Combine all patterns with bout type
patterns = [
    ('FIRST_ORDER', *first_order[0]),
    ('SECOND_ORDER', *second_order[0]),
    ('SECOND_ORDER', *second_order[1]),
    ('SECOND_ORDER', *second_order[2]),
    ('THIRD_ORDER', *third_order[0]),
    ('THIRD_ORDER', *third_order[1]),
    ('THIRD_ORDER', *third_order[2])
]

In [13]:
import pandas as pd
import datetime
import statistics


bout_df_list = []
for id_pick in sleep_id_array:
    print(id_pick)
    sleep_df_idPick = sleep_df1[sleep_df1['patient_id'] == id_pick] 
    
    date_array = sleep_df_idPick.sleep_date.unique()
    for target_date in date_array:

        # in each day
        filtered_df = sleep_df_idPick[sleep_df_idPick['sleep_date'] == target_date]
        
        # Assuming your dataframe is called sleep_df_idPick
        # First, make sure timestamp is a datetime object
        filtered_df['timestamp'] = pd.to_datetime(filtered_df['timestamp'])
        
        # Sort by timestamp
        filtered_df = filtered_df.sort_values('timestamp').reset_index(drop=True)
        
##################### calculate bout duration ##########################
        # Identify chunks of consecutive same state
        filtered_df['state_change'] = (filtered_df['state'] != filtered_df['state'].shift()).cumsum()
        
        # Group by each chunk
        chunk_durations = filtered_df.groupby(['state_change', 'state']).agg(
            start_time=('timestamp', 'first'),
            end_time=('timestamp', 'last')
        ).reset_index()
        chunk_durations = chunk_durations.dropna(subset=['start_time']).reset_index()
        
        # Calculate duration in minutes per chunk
        chunk_durations['duration_min'] = (chunk_durations['end_time'] - chunk_durations['start_time']).dt.total_seconds() / 60
        
            
     # Detect bouts
        bouts = []
        i = 0
        while i < len(chunk_durations):
            matched = False
            for bout_type, seq, valid_next in patterns:
                n = len(seq)
                if i + n <= len(chunk_durations) and list(chunk_durations['state'][i:i+n]) == seq:
                    # Check next state (if it exists)
                    if i + n >= len(chunk_durations) or chunk_durations['state'][i + n] in valid_next:
                        bouts.append({
                            'patient_id' : id_pick,
                            'sleep_date' : target_date,
                            'start_idx': i,
                            'end_idx': i + n - 1,
                            'bout_type': bout_type,
                            'seq_type': seq,
                            'total_duration': chunk_durations['duration_min'][i:i+n].sum(),
                            'start_time': chunk_durations['start_time'][i],
                            'end_time': chunk_durations['end_time'][i+n-1]
                        })
                        i += n
                        matched = True
                        break
            if not matched:
                i += 1  # Move forward if no pattern matched
        
        # Convert result to DataFrame
        bout_df = pd.DataFrame(bouts)
        bout_df_list.append(bout_df)

In [14]:
final_df = pd.concat(bout_df_list, ignore_index=True)
# drop those durations = 0 (only one state without knowing how long it lasts)
final_df = final_df[final_df['total_duration'] != 0]
final_df.to_csv('../output/data_cleaned/sleep_night_bouts_detailed.csv', index=False)
final_df

In [15]:
simple_df = final_df.groupby(['patient_id', 'sleep_date']).agg(
    bout_duration_list=('total_duration', list),
    bout_duration_mean=('total_duration', 'mean'),
    bout_number=('total_duration', 'count')
).reset_index()
simple_df.to_csv('../output/data_cleaned/sleep_night_bouts_summary.csv', index=False)

simple_df